# Portrait Styling Lab — Thin Colab A/B Benchmark Runner (SFW Only)

All pipeline code lives in GitHub (`lifergie6-a11y/portrait-styling-lab`). This notebook only:
1. Mounts Google Drive into an isolated SFW workspace (`MyDrive/portrait-styling-lab/`)
2. Clones / updates the repo & installs dependencies
3. *(Optional, run on free CPU runtime first)* Caches model weights onto Google Drive
4. Runs offline batch A/B benchmarking (`FireRed-Image-Edit 1.1` vs `Qwen-Image-2.1`) and saves a static `report.html` (no public web tunnel / ToS-safe)

In [ ]:
#@title 1. Mount Google Drive & Configure Isolated SFW Workspace
import os
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

WORKSPACE = Path('/content/drive/MyDrive/portrait-styling-lab')
HF_CACHE = WORKSPACE / 'hf_cache'
DATASET_DIR = WORKSPACE / 'dataset'
RUNS_DIR = WORKSPACE / 'runs'

for d in (WORKSPACE, HF_CACHE, DATASET_DIR, RUNS_DIR):
    d.mkdir(parents=True, exist_ok=True)

os.environ['HF_HOME'] = str(HF_CACHE)
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
print('Workspace ready:', WORKSPACE)

In [ ]:
#@title 2. Clone / Update Repository & Install Dependencies
REPO_URL = 'https://github.com/lifergie6-a11y/portrait-styling-lab.git'
REPO_DIR = Path('/content/portrait-styling-lab')

if REPO_DIR.exists():
    !git -C /content/portrait-styling-lab pull --ff-only
else:
    !git clone {REPO_URL} /content/portrait-styling-lab

!pip install -q -r /content/portrait-styling-lab/requirements-colab.txt
!pip install -q -e /content/portrait-styling-lab
!pstyle doctor

In [ ]:
#@title 3. (Recommended on FREE CPU Runtime First) Pre-cache Weights to Drive
CACHE_MODELS = 'firered-1.1,qwen-2.1'  #@param {type:'string'}
!python3 /content/portrait-styling-lab/scripts/colab_cache_models.py \
    --models "{CACHE_MODELS}" \
    --cache-dir "{HF_CACHE}"

In [ ]:
#@title 4. Auto-generate manifest.json if you placed test portraits in MyDrive/portrait-styling-lab/dataset/images/
IMAGES_DIR = DATASET_DIR / 'images'
MANIFEST_PATH = DATASET_DIR / 'manifest.json'

if not MANIFEST_PATH.exists():
    IMAGES_DIR.mkdir(parents=True, exist_ok=True)
    print(f'Upload 10-30 consented test portraits to {IMAGES_DIR}, then re-run this cell.')
    if any(IMAGES_DIR.iterdir()):
        !python3 /content/portrait-styling-lab/scripts/create_sample_dataset.py \
            --images-dir "{IMAGES_DIR}" \
            --out "{MANIFEST_PATH}"
else:
    print('Using existing manifest:', MANIFEST_PATH)

In [ ]:
#@title 5. Run A/B Foundation Model Benchmark (Attach L4 / A100 / G4 GPU)
MODELS = 'firered-1.1,qwen-2.1'  #@param {type:'string'}
RUN_NAME = 'ab_v0'  #@param {type:'string'}
NUM_SEEDS = 2  #@param {type:'integer'}
QUANTIZATION = 'auto'  #@param ['auto', 'none', 'nf4']
LIMIT = 0  #@param {type:'integer'}

OUT_DIR = RUNS_DIR / RUN_NAME
limit_flag = f'--limit {LIMIT}' if LIMIT > 0 else ''

!pstyle benchmark \
    --manifest "{MANIFEST_PATH}" \
    --models "{MODELS}" \
    --num-seeds {NUM_SEEDS} \
    --quantization {QUANTIZATION} \
    --cache-dir "{HF_CACHE}" \
    --output-dir "{OUT_DIR}" \
    {limit_flag}

print('Static HTML report saved at:', OUT_DIR / 'report.html')

In [ ]:
#@title 6. Download Static HTML A/B Report for Local / Expert Blind Review
from google.colab import files
report_file = RUNS_DIR / RUN_NAME / 'report.html'
if report_file.exists():
    files.download(str(report_file))
else:
    print('Report not found yet:', report_file)